In [0]:
from pyspark.sql import functions as F

# De procesos solo traemos las columnas que necesitamos, para no chocar con las de adjudicaciones
procesos = (
    spark.table("oece.silver.procesos")
    .filter("categoria = 'works'")
    .select(
        "ocid", "nomenclatura", "descripcion", "tipo_procedimiento",
        "id_entidad", "entidad", "departamento", "fecha_publicacion",
    )
)
adj = spark.table("oece.silver.adjudicaciones")

df_gold = (
    adj.join(procesos, "ocid")
    .select(
        "id_adjudicacion",
        "ocid",
        "nomenclatura",
        "descripcion",
        "tipo_procedimiento",
        "id_entidad",
        "entidad",
        "departamento",
        "ruc_proveedor",
        "proveedor",
        "fecha_publicacion",
        "fecha_adjudicacion",
        F.date_format("fecha_adjudicacion", "yyyy-MM").alias("mes_adjudicacion"),
        "moneda",
        "monto_adjudicado",
        "monto_adjudicado_pen",
    )
)

df_gold.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("oece.gold.obras_adjudicadas")

gold = spark.table("oece.gold.obras_adjudicadas")
print("filas:", gold.count())

display(
    gold.groupBy("departamento")
    .agg(
        F.count("*").alias("adjudicaciones"),
        F.round(F.sum("monto_adjudicado_pen") / 1e6, 1).alias("monto_millones_pen"),
    )
    .orderBy(F.desc("monto_millones_pen"))
    .limit(10)
)

filas: 2359


departamento,adjudicaciones,monto_millones_pen
LIMA,197,1969.1
PIURA,74,1194.2
CAJAMARCA,251,687.8
ANCASH,377,553.1
JUNIN,195,526.2
AREQUIPA,98,334.8
AYACUCHO,107,326.0
HUANUCO,117,319.2
LA LIBERTAD,167,306.1
UCAYALI,49,269.5
